# Прогноз перспективных площадей — таргет = КРИТЕРИАЛЬНЫЙ анализ (не геохимия)

**Задача.** Научить ML-классификатор воспроизводить/предсказывать перспективные
площади критериального анализа ГИС Интегро («Таксономия по критериям») по
геофизическим, дистанционным (Landsat-7) и структурным признакам из
`dataset_v1`. Геохимия в качестве таргета **не используется** — более того,
геохимические слои намеренно исключены из признаков (стоп-лист при сборке
датасета, см. `data/dataset_v1/dataset_v1_sources.md`).

**Таргет.** Нативный результат критериального расчёта
`data/Gis-integro/Расчет/prognoz.prognoz.property` (22 946 значений на сетке
154×149, шаг 500 м). В шкале ГИС Интегро **меньше = перспективнее** (мера
Плюты — расстояние до эталона-минимума). Бинаризация: положительный класс =
верхние **15 %** наиболее перспективных ячеек (порог настраивается в
`config.CRITERIAL_TOP_FRAC`).

**Признаки.** НЕЗАВИСИМЫЙ от критериального набор — **28 признаков**:
гравика/магнитка и их трансформации (17), 7 каналов Landsat-7, рельеф,
`dist_dnl`/`dist_dnara`, маска свиты. Факторные признаки
(`dist_facies/struct/magm/paleo/tect1/tect2`, `dens_tect/dens_magm`), из которых
сам критериальный и построен, ИСКЛЮЧЕНЫ через `config.CRITERIAL_EXCLUDE_FEATURES`
— иначе модель тривиально восстанавливает формулу критериального (утечка, ROC до
0.99). Здесь проверяется, воспроизводима ли перспективность по независимым
данным (ожидаемо ROC ≈ 0.94, lift@10% ≈ 5×).

**Валидация.** Out-of-fold по пространственным блокам (`GroupKFold`, блок
15×15 ячеек) — соседние коррелированные ячейки не попадают одновременно в
train и test. Метрики: ROC-AUC, PR-AUC, lift@N %.

In [ ]:
import sys, pathlib
# запуск из notebook/ — добавляем корень репозитория в путь
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from src.criterial_data import load_criterial_dataset
from src import criterial_model as cm
from src import config

print('top_frac =', config.CRITERIAL_TOP_FRAC, '| блок CV =', config.CRITERIAL_BLOCK_CELLS,
      'ячеек | фолдов =', config.CRITERIAL_N_SPLITS)

## 1. Загрузка датасета и построение критериального таргета

In [ ]:
data = load_criterial_dataset()
print(f'Ячеек: {len(data.y)} | признаков: {len(data.feature_cols)} | '
      f'сетка {data.grid_shape}')
print(f'Доля положительного класса: {data.y.mean():.3f} '
      f'(порог критериального ≤ {data.crit[data.y==1].max():.4f})')
print('Признаки:', data.feature_cols)

### Критериальная поверхность и бинарный таргет

In [ ]:
crit_grid = np.full(data.grid_shape, np.nan)
tgt_grid  = np.full(data.grid_shape, np.nan)
r = data.frame['row'].to_numpy().astype(int); k = data.frame['col'].to_numpy().astype(int)
crit_grid[r, k] = data.crit
tgt_grid[r, k]  = data.y

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im0 = ax[0].imshow(crit_grid, cmap='viridis_r', origin='upper')  # _r: тёмное = перспективно
ax[0].set_title('Критериальный анализ ГИС Интегро\n(меньше = перспективнее)')
fig.colorbar(im0, ax=ax[0], shrink=0.7)
ax[1].imshow(tgt_grid, cmap='Reds', origin='upper')
ax[1].set_title(f'Бинарный таргет: топ-{int(config.CRITERIAL_TOP_FRAC*100)}% перспективных')
for a in ax: a.set_xlabel('col (X, З→В)'); a.set_ylabel('row (Y, С→Ю)')
plt.tight_layout(); plt.show()

## 2. Честная пространственная кросс-валидация

Out-of-fold прогноз по пространственным блокам. **Время:** ~1–3 мин
(RF 300 + GB 150 деревьев × 5 фолдов; на 2 ядрах дольше).

In [ ]:
res = cm.spatial_cv(data)
print(f'ROC-AUC (OOF): {res.roc_auc:.4f}')
print(f'PR-AUC  (OOF): {res.pr_auc:.4f}')
print('\nLift@N% (во сколько раз доля перспективных в топ-N% прогноза выше базовой):')
for a, v in res.lift.items():
    print(f'  lift@{int(a*100):>2d}% : {v:.2f}x')
print('\nПо фолдам (ROC-AUC):', [round(f['roc_auc'], 3) for f in res.per_fold])

### Кривая обнаружения (lift) и ROC

In [ ]:
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
# lift-кривая: доля найденных перспективных vs доля обследованной площади
order = np.argsort(-res.oof_score)
y_sorted = data.y[order]
frac_area = np.arange(1, len(y_sorted)+1) / len(y_sorted)
frac_found = np.cumsum(y_sorted) / data.y.sum()
ax[0].plot(frac_area, frac_found, label='ML (критериальный таргет)')
ax[0].plot([0,1],[0,1],':',color='gray',label='случайный')
ax[0].set_xlabel('доля обследованной площади'); ax[0].set_ylabel('доля найденных перспективных')
ax[0].set_title('Кривая обнаружения'); ax[0].legend()
fpr, tpr, _ = roc_curve(data.y, res.oof_score)
ax[1].plot(fpr, tpr, label=f'AUC={res.roc_auc:.3f}'); ax[1].plot([0,1],[0,1],':',color='gray')
ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR'); ax[1].set_title('ROC'); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Важности признаков

In [ ]:
model = cm.fit_full(data)
imp = cm.feature_importance(model, data)
names = [n for n, _ in imp][:15][::-1]; vals = [v for _, v in imp][:15][::-1]
plt.figure(figsize=(8, 6)); plt.barh(names, vals)
plt.title('Топ-15 признаков (ансамбль RF+GB)'); plt.xlabel('важность'); plt.tight_layout(); plt.show()
for n, v in imp[:10]: print(f'{n:22s} {v:.3f}')

## 4. Карта прогноза по всей сетке

Вероятность принадлежности к перспективному (по критериальному анализу) классу,
предсказанная моделью на признаках `dataset_v1`. Сравнение с исходной
критериальной поверхностью — качественная заверка.

In [ ]:
pred_grid = cm.predict_grid(model, data)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im0 = ax[0].imshow(pred_grid, cmap='magma', origin='upper')
ax[0].set_title('ML-прогноз: P(перспективно)'); fig.colorbar(im0, ax=ax[0], shrink=0.7)
im1 = ax[1].imshow(crit_grid, cmap='viridis_r', origin='upper')
ax[1].set_title('Критериальный анализ (таргет)'); fig.colorbar(im1, ax=ax[1], shrink=0.7)
for a in ax: a.set_xlabel('col'); a.set_ylabel('row')
plt.tight_layout(); plt.show()

---
### Замечание о постановке

Факторные признаки (`dist_facies/struct/magm/paleo/tect1/tect2`,
`dens_tect/dens_magm`) исключены по умолчанию — они входы самой формулы
критериального анализа, и с ними метрики тавтологичны (ROC ≈ 0.99). Текущий
результат (ROC ≈ 0.94, lift@10% ≈ 5×) получен на НЕЗАВИСИМЫХ данных и показывает
реальную геофизическую подпись критериальной перспективности.

**Ограничение:** метрики — на отложенных блоках ТОЙ ЖЕ карты. Перенос на другую
территорию требует отдельного теста на втором листе (обучение на A, замер на B).

Вернуть факторные признаки для сравнения — очистить
`config.CRITERIAL_EXCLUDE_FEATURES` и перезапустить.